# 07 — The Pauli Gates

Meet $X$, $Y$, $Z$ — the three most important single-qubit gates — and their double life as both operations *and* observables.

**Prerequisites:** `01/06_gates_as_unitary_evolution`.

**What you'll be able to do**
- Apply the Pauli gates: $X$ flips a bit, $Z$ flips a phase, $Y$ does both.
- See each Pauli as a $180^\circ$ rotation of the Bloch sphere, and check $X^2 = Y^2 = Z^2 = I$.
- Recognise $X, Y, Z$ as the **observables** whose averages are the Bloch coordinates $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$.

In `01/06` you learned every gate is a unitary rotation. Now we name the three that show up everywhere. They carry a beautiful surprise: the same three matrices are also the measurements behind the Bloch picture — the operator/observable bridge the whole course quietly relies on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, bloch_vector
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z, IDENTITY, apply_gate, is_unitary, expectation

np.random.seed(42)
viz.use_course_style()

## 1. X flips the bit, Z flips the phase, Y does both

$$ X = \begin{pmatrix}0&1\\1&0\end{pmatrix},\qquad Y = \begin{pmatrix}0&-i\\i&0\end{pmatrix},\qquad Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}. $$

$X$ is the quantum **NOT**: it swaps $|0\rangle$ and $|1\rangle$. $Z$ leaves $|0\rangle$ alone but multiplies $|1\rangle$ by $-1$ — a **phase flip**, which turns $|+\rangle$ into $|-\rangle$. $Y$ combines the two.

In [ ]:
print("X|0> =", np.round(apply_gate(PAULI_X, KET_0), 3), " (= |1>: bit flip)")
print("X|1> =", np.round(apply_gate(PAULI_X, KET_1), 3), " (= |0>)")
print("Z|0> =", np.round(apply_gate(PAULI_Z, KET_0), 3), " (unchanged)")
print("Z|1> =", np.round(apply_gate(PAULI_Z, KET_1), 3), " (picks up -1)")
print("Z|+> =", np.round(apply_gate(PAULI_Z, KET_PLUS), 3), " (= |->: phase flip swaps + and -)")
print("Y|0> =", np.round(apply_gate(PAULI_Y, KET_0), 3), " (= i|1>: bit + phase)")

**Read the output.** $X$ exchanges the two basis states; $Z$ leaves $|0\rangle$ unchanged but stamps a $-1$ on $|1\rangle$, which is exactly what carries $|+\rangle \to |-\rangle$ (the phase flip from `01/02`, now delivered by a gate). $Y$ flips the bit *and* adds a phase. In a Qiskit circuit these are `qc.x(0)`, `qc.y(0)`, `qc.z(0)`.

## 2. Each Pauli is a half-turn of the Bloch sphere

Every Pauli is unitary (a valid gate) and also **its own inverse**: $X^2 = Y^2 = Z^2 = I$. Geometrically each is a $180^\circ$ rotation about its axis — apply it twice and you return to where you started. Here is $X$ taking the north pole $|0\rangle$ to the south pole $|1\rangle$.

In [ ]:
for name, P in [("X", PAULI_X), ("Y", PAULI_Y), ("Z", PAULI_Z)]:
    print(f"{name}: unitary={is_unitary(P)},  {name}^2 = I ? {np.allclose(P @ P, IDENTITY)}")

fig = viz.plot_bloch(apply_gate(PAULI_X, KET_0), title=r"$X|0\rangle = |1\rangle$: a half-turn about the x-axis")
plt.show()

**Read the figure.** $X$ sends $|0\rangle$ (north pole) straight to $|1\rangle$ (south pole) — a $180^\circ$ rotation about the $x$-axis. Each Pauli does the same about its own axis, and squaring to the identity is the algebraic echo of "a half-turn twice is a full turn".

## 3. The double life: Paulis are also observables

Here is the surprise. The very same matrices are **Hermitian** ($P^\dagger = P$), so each is a legitimate *observable* — a measurable quantity with outcomes $\pm 1$. Their averages on a state are precisely its Bloch coordinates from `01/03`:

$$ \big(\langle X\rangle,\ \langle Y\rangle,\ \langle Z\rangle\big) = \big(\langle\psi|X|\psi\rangle,\ \langle\psi|Y|\psi\rangle,\ \langle\psi|Z|\psi\rangle\big) = \text{Bloch vector of } |\psi\rangle. $$

(The average $\langle A\rangle = \langle\psi|A|\psi\rangle$ is the **expectation value**, developed fully in `01/10`.) So $X, Y, Z$ are gates you *apply* and, read the other way, the measurements that *locate* a state on the sphere.

In [ ]:
psi = qubit_state(theta=2 * np.pi / 3, phi=np.pi / 4)
exps = [expectation(psi, P) for P in (PAULI_X, PAULI_Y, PAULI_Z)]
print("(<X>, <Y>, <Z>) =", np.round(exps, 3))
print("bloch_vector    =", np.round(bloch_vector(psi), 3), " (identical)")

fig, ax = plt.subplots(figsize=(7.0, 4.2))
labels = [r"$\langle X\rangle$", r"$\langle Y\rangle$", r"$\langle Z\rangle$"]
ax.bar(labels, exps, color=[COLORS["source"], COLORS["flow"], COLORS["quantum"]])
ax.axhline(0, color=COLORS["grid"], lw=1)
ax.set_ylim(-1, 1)
ax.set_ylabel("expectation value (= Bloch coordinate)")
ax.set_title("Pauli expectations are the coordinates of the state", pad=12)
plt.show()

**Read the figure.** The three Pauli expectation values reproduce the state's Bloch vector exactly — bar for bar. This is the operator/observable duality in one picture: to *move* a qubit you apply a Pauli as a gate; to *find* a qubit you measure the Paulis and read off $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$. It is also the recipe `01/15` uses to reconstruct a density matrix from a real device.

## Your turn

1. **Phase flip both ways.** Compute $Z|-\rangle$ and confirm $Z$ swaps $|+\rangle \leftrightarrow |-\rangle$. Why does $Z$ leave the Bloch $z$-axis fixed but flip the equator?
2. **Y from X and Z.** Verify $Y = iXZ$ as matrices. (The Paulis are deeply interlinked — this relation is one face of it.)
3. **Eigen-meaning.** Find the eigenvectors of $X$ and their eigenvalues. (You should recover $|+\rangle, |-\rangle$ with $\pm 1$.) This is why "measuring $X$" means asking whether a state is $|+\rangle$ or $|-\rangle$ — the basis change you build in `01/08`.

## What you built

- You applied the Pauli gates: $X$ (bit flip), $Z$ (phase flip, $|+\rangle\to|-\rangle$), $Y$ (both).
- You saw each Pauli as a $180^\circ$ Bloch rotation and confirmed $X^2 = Y^2 = Z^2 = I$.
- You uncovered their double life: as Hermitian *observables*, their expectations $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ are the Bloch coordinates of the state.

Three matrices, two roles, and the bridge between "operate" and "measure". This duality is one of the most reused ideas in the whole course — and now it is yours.

## What's next

There is one more single-qubit gate every circuit opens with — the Hadamard. In `01/08_hadamard_and_measurement_bases` we use it to build $|+\rangle$ and, turning it around, to *measure in the $X$ basis* — finally explaining the `h` and `sdg` that `01/11` and `01/15` use.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 1.3.1, 4.2, Cambridge University Press (2010).
- W. Pauli, "Zur Quantenmechanik des magnetischen Elektrons", *Zeitschrift für Physik* 43, 601-623 (1927). DOI:10.1007/BF01397326

**Previous:** `notebooks/01_QuantumFoundations/06_gates_as_unitary_evolution.ipynb` · **Next:** `notebooks/01_QuantumFoundations/08_hadamard_and_measurement_bases.ipynb`